# 検出器データ解析ノートブック (DMA 版) — `.npz` ファイル

このノートブックは、Red Pitaya で取得した **`.npz` ファイル** を解析するためのものです。`detector-acquisition-dma.ipynb` の **Option B (DMA)** で保存された **全波形データ** を対象にしています。

## 何が入っているデータ？

`.npz` ファイルは「複数の numpy 配列を 1 つの zip にまとめたもの」で、配列にはそれぞれ **キー名** が付いています。

| キー名 | 形 | 意味 |
|---|---|---|
| `ch1` | `(N_shots, samples_per_shot)` | ch1 の波形 [V] (1 行 = 1 ショット) |
| `ch2` | `(N_shots, samples_per_shot)` | ch2 の波形 [V] |
| `totaltime` | `(N_shots,)` | 計測開始からの経過時間 [秒] |
| `deadtime`  | `(N_shots,)` | 累積デッドタイム [秒] |

加えて、**同じファイル名の `.json`** に取得条件 (サンプリング周期、トリガ条件、入力ジャンパ設定など) が入っています。

## このノートブックで体験すること

1. **波形を表示する** — 1 イベントを実際にプロット。
2. **平均化** — 全イベントを平均してノイズを減らす。
3. **ピーク時刻と時間差** — 2 ch のピーク位置の差を求める。
4. **補間による平滑化** — 離散サンプルの間を埋めて滑らかにする。
5. **相互相関** — 2 ch の波形がどれだけずれているかを計算する。
6. **遅延 (時間差) の精密評価** — 相互相関のピークから真の時間差を求める。

> **読み方のコツ**: 波形は 1 サンプル = 8 ナノ秒 (=125 MSPS) で取られています。デシメーションが 1 でない場合はもう少し長くなります。

## 0. ライブラリの準備

このノートブックでは 2 つのライブラリを使います (どちらも Red Pitaya のシステム Python に標準で入っているはず)。

- **numpy** — 配列計算 + `.npz` の読み書き (`np.load`)
- **plotly** — グラフ描画

メタ情報の `.json` を読むのには、Python 標準の `json` モジュールを使うので追加インストールはありません。

もし `plotly` が入っていなければ、下のセルの先頭の `#` を外して 1 度だけ実行してください。

In [ ]:
# import sys, subprocess
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "numpy", "plotly"])

In [ ]:
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("numpy", np.__version__)

## 1. データの読み込み

### 1.1 `./data/` の中で **一番新しい** `.npz` ファイルを自動的に探す

特定のファイルを開きたいときは、自動検出を使わず `npz_path = Path('./data/2026-04-28-140500.npz')` のように直接指定してもかまいません。

In [ ]:
def find_latest(ext: str, folder: str = "./data") -> Path:
    '''folder 内で拡張子 ext を持つファイルのうち、更新時刻が最新のものを返す。'''
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(
            f"{folder} が見つかりません。取得用ノートブックでデータを取った後に実行してください。"
        )
    candidates = sorted(folder.glob(f"*{ext}"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"{folder} の中に *{ext} ファイルが見つかりません。")
    return candidates[0]

npz_path = find_latest(".npz")
print("読み込むファイル:", npz_path)

### 1.2 中身を確認

`.npz` には 4 つの numpy 配列(`ch1`, `ch2`, `totaltime`, `deadtime`)が入っていて、それぞれの **キー名** で取り出せます。
取得条件のメモは **同じ名前の `.json`** にあるので、こちらも読みましょう。

In [ ]:
data = np.load(npz_path)

print("=== .npz の中のキー (配列名) ===")
for name in data.files:
    arr = data[name]
    print(f"  {name:12s} shape={arr.shape}  dtype={arr.dtype}")

meta_path = npz_path.with_suffix(".json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print("\n=== 取得条件 (.json) ===")
    for k, v in meta.items():
        print(f"  {k:24s}: {v}")
else:
    meta = {}
    print("\n(対応する .json が見つかりませんでした。サンプリング周期はデフォルト値 8 ns を使います。)")

### 1.3 配列を取り出す

`np.load` で開いた `.npz` は **キーごとに必要なときだけ読み込む**「遅延ロード」型ですが、ここでは後段の処理でまとめて使うので **全部 numpy 配列に展開** します。
**もしファイルが極端に大きい場合** (メモリより大きい等) は、各ショットを `data["ch1"][i]` のように 1 行ずつ読む方が安全です。

In [ ]:
ch1 = data["ch1"]   # shape: (N_shots, samples_per_shot)
ch2 = data["ch2"]
totaltime = data["totaltime"]
deadtime  = data["deadtime"]

samples_per_shot = int(meta.get("samples_per_shot", ch1.shape[1]))
# サンプリング周期: メタに sample_dt_s があれば優先、なければ dec=1 (125 MSPS) で 8 ns
dt_s = float(meta.get("sample_dt_s", 8e-9))
known_delay = meta.get("known_delay_samples", None)  # フィクスチャ用 (本物の取得では存在しない)

n_shots = ch1.shape[0]
print(f"ショット数        : {n_shots}")
print(f"1 ショットの長さ  : {samples_per_shot} samples")
print(f"サンプリング周期  : {dt_s*1e9:.2f} ns/sample  ({1/dt_s/1e6:.2f} MSPS)")
print(f"全波形メモリ      : {ch1.nbytes/1e6:.2f} MB (ch1) + {ch2.nbytes/1e6:.2f} MB (ch2)")

## 2. 波形の表示

### 2.1 1 イベント目を見てみる

横軸はサンプル番号ではなく **時間 [μs]** に変換しました (`t = サンプル番号 × dt_s × 1e6`)。
これで「波形の幅は何ナノ秒くらい？」が一目で分かります。


In [ ]:
t_us = np.arange(samples_per_shot) * dt_s * 1e6  # [μs]

event_idx = 0  # ← 別のイベントを見たいときはここを変えてください
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_us, y=ch1[event_idx], mode="lines", name=f"ch1 (event {event_idx})"))
fig.add_trace(go.Scatter(x=t_us, y=ch2[event_idx], mode="lines", name=f"ch2 (event {event_idx})"))
fig.update_layout(xaxis_title="time [μs]", yaxis_title="voltage [V]",
                  width=1000, height=400, title_text=f"波形 (event {event_idx})")
fig.show()


### 2.2 別のイベントも見比べる

`event_idx` を変えて何度か実行してみよう。波形の **形** はだいたい同じ？ それともショットごとに違いますか？


In [ ]:
# 4 個のイベントを並べて表示
indices = [0, 1, 2, 3] if n_shots >= 4 else list(range(n_shots))
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"event {i}" for i in indices])
for k, i in enumerate(indices):
    r, c = k // 2 + 1, k % 2 + 1
    fig.add_trace(go.Scatter(x=t_us, y=ch1[i], mode="lines", name="ch1",
                             line=dict(color="royalblue"), showlegend=(k == 0)), row=r, col=c)
    fig.add_trace(go.Scatter(x=t_us, y=ch2[i], mode="lines", name="ch2",
                             line=dict(color="orange"),    showlegend=(k == 0)), row=r, col=c)
fig.update_xaxes(title_text="time [μs]")
fig.update_yaxes(title_text="voltage [V]")
fig.update_layout(width=1100, height=600, title_text="複数イベントの波形")
fig.show()


## 3. 全波形の平均化 — ノイズを減らす

ノイズはショットごとにランダムに揺れますが、**信号は毎回ほぼ同じ場所に出る** ので、
全イベントを平均すると **信号は残ったままノイズだけが減る** という性質があります。

理論的には、N 個のショットを平均するとノイズは **√N 倍小さく** なります (例: 100 ショット平均でノイズは 1/10)。


In [ ]:
ch1_mean = ch1.mean(axis=0)
ch2_mean = ch2.mean(axis=0)

fig = make_subplots(rows=1, cols=2, subplot_titles=("ch1: 1 イベント vs 平均", "ch2: 1 イベント vs 平均"))
fig.add_trace(go.Scatter(x=t_us, y=ch1[0],   mode="lines", name="event 0",
                         line=dict(color="lightblue")),  row=1, col=1)
fig.add_trace(go.Scatter(x=t_us, y=ch1_mean, mode="lines", name=f"mean (N={n_shots})",
                         line=dict(color="navy", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_us, y=ch2[0],   mode="lines", name="event 0",
                         line=dict(color="moccasin")), row=1, col=2)
fig.add_trace(go.Scatter(x=t_us, y=ch2_mean, mode="lines", name=f"mean (N={n_shots})",
                         line=dict(color="darkorange", width=2)), row=1, col=2)
fig.update_xaxes(title_text="time [μs]"); fig.update_yaxes(title_text="voltage [V]")
fig.update_layout(width=1100, height=420)
fig.show()

# ノイズ (ベースラインの標準偏差) が平均で減ったかを定量確認
# ベースラインは波形の前半 1/4 に取る (信号がまだ来ていない部分)
nb = samples_per_shot // 4
noise_one  = float(np.std(ch1[0, :nb]))
noise_mean = float(np.std(ch1_mean[:nb]))
print(f"ch1 ベースラインノイズ (1 イベント): {noise_one*1e3:.2f} mV")
print(f"ch1 ベースラインノイズ (平均後)     : {noise_mean*1e3:.2f} mV")
print(f"理論予測 (1/√N = 1/√{n_shots}): {noise_one/np.sqrt(n_shots)*1e3:.2f} mV")


## 4. ピーク時刻と 2 ch の時間差

検出器に同じ粒子が入っても、ch1 と ch2 でわずかに **到達時間がずれる** ことがあります。
最初のステップとして、各イベントの `argmax` (最大値の位置) を使って素朴に時間差を測ってみましょう。


In [ ]:
peak_ch1 = np.argmax(ch1, axis=1)            # samples
peak_ch2 = np.argmax(ch2, axis=1)
delta_samples = peak_ch2 - peak_ch1
delta_ns      = delta_samples * dt_s * 1e9   # [ns]

fig = go.Figure()
fig.add_trace(go.Histogram(x=delta_ns, nbinsx=max(15, int(np.sqrt(len(delta_ns)))),
                           name="ch2 - ch1 peak"))
fig.update_layout(xaxis_title="ピーク位置の差 ch2 - ch1 [ns]", yaxis_title="件数",
                  width=900, height=380, title_text="2 ch のピーク時刻の差 (素朴な argmax)")
fig.show()

print(f"中央値: {np.median(delta_ns):.2f} ns")
print(f"平均  : {np.mean(delta_ns):.2f} ns")
print(f"標準偏差: {np.std(delta_ns):.2f} ns")


**注意**: `argmax` で求めるピーク位置は **整数サンプル** までしか分からないので、
ピーク差は dt = 8 ns 単位でしか測れません。これより細かい精度が欲しいときは、
このあと **§ 7 (相互相関 + 放物線フィット)** で **サブサンプル精度** に挑戦します。


## 5. 補間による平滑化

離散サンプルでカクカクした波形を「滑らかに」見せるには、サンプルの間を **補間 (interpolation)** で埋めます。
ここでは最も基本的な **線形補間** (`numpy.interp`) で 10 倍にアップサンプリングしてみましょう。

> **何のためにやるのか?**
> ピーク位置を細かく測ったり、ノイズの形を観察したりするときに、サンプル間隔より細かい時間スケールで波形を「のぞき見」することができます。
> ただし補間で **新しい情報が増えるわけではない** ので、ナイキスト周波数より高い周波数成分は復元できません。


In [ ]:
# ピーク周辺の窓を切り出して補間する
event_idx = 0
peak_pos = int(np.argmax(ch1[event_idx]))
half = 30
lo = max(0, peak_pos - half)
hi = min(samples_per_shot, peak_pos + half)

x_orig = np.arange(lo, hi)
y_orig = ch1[event_idx, lo:hi]

# 10 倍に細かい x 軸
x_fine = np.linspace(lo, hi - 1, 10 * (hi - lo))
y_fine = np.interp(x_fine, x_orig, y_orig)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_orig * dt_s * 1e6, y=y_orig, mode="markers+lines",
                         name="元データ (8 ns 間隔)", marker=dict(size=8, color="royalblue")))
fig.add_trace(go.Scatter(x=x_fine * dt_s * 1e6, y=y_fine, mode="lines",
                         name="線形補間 ×10", line=dict(color="firebrick", width=1)))
fig.update_layout(xaxis_title="time [μs]", yaxis_title="voltage [V]",
                  width=1000, height=420, title_text=f"ピーク周辺の補間 (event {event_idx})")
fig.show()


## 6. 相互相関関数

**相互相関 (cross-correlation)** は、2 つの波形を **少しずつずらしながら掛け合わせる** 操作です。
2 つの波形が「ある時間ずらすとぴったり重なる」とき、その **ずらし量 (ラグ τ)** で相関値が最大になります。

ここでは、**「ch2 が ch1 より τ だけ遅れている」** のとき τ が **正** になるようにラグを定義します。

$$ C(\tau) = \sum_{n} \text{ch1}(n) \cdot \text{ch2}(n + \tau) $$

> **numpy 実装の注意**: `np.correlate(a, b, mode="full")` の返す配列は **a と b の順序によってラグの向きが変わる** という少しトリッキーな仕様です。
> 上の τ 定義 (正 = ch2 が遅延) に合わせるため、コードでは **`np.correlate(ch2, ch1, ...)`** と引数を入れ替えています (こうすると配列のインデックスが `-N+1, ..., 0, ..., N-1` の順で τ と素直に一致します)。


In [ ]:
# 1 イベントで相互相関を計算
event_idx = 0
a = ch1[event_idx] - ch1[event_idx].mean()    # ベースラインを引く (DC 成分でピークがぼけるのを防ぐ)
b = ch2[event_idx] - ch2[event_idx].mean()
N = len(a)

# np.correlate(b, a, ...) を使うことで、lags[argmax] > 0  →  ch2 が ch1 より遅れている、という素直な読みになる
corr = np.correlate(b, a, mode="full")           # 長さ 2N-1
lags = np.arange(-N + 1, N)                      # -N+1, ..., 0, ..., N-1
lag_ns = lags * dt_s * 1e9

# 見やすさのため、ピーク周辺だけ表示
view = np.abs(lag_ns) < 200  # ±200 ns
fig = go.Figure()
fig.add_trace(go.Scatter(x=lag_ns[view], y=corr[view], mode="lines", name="cross-corr"))
fig.update_layout(xaxis_title="ラグ τ [ns]   (正 = ch2 が ch1 より遅れている)",
                  yaxis_title="相関値",
                  width=1000, height=380, title_text="相互相関関数 (event 0, ±200 ns)")
fig.show()

peak_lag_samples = int(lags[np.argmax(corr)])
peak_lag_ns      = peak_lag_samples * dt_s * 1e9
print(f"相関のピーク位置: τ = {peak_lag_samples} サンプル ({peak_lag_ns:.2f} ns)")
if peak_lag_ns >= 0:
    print(f"→ ch2 は ch1 より約 {peak_lag_ns:.1f} ns 遅れている、と読めます。")
else:
    print(f"→ ch2 は ch1 より約 {-peak_lag_ns:.1f} ns 早い、と読めます。")


## 7. 相互相関を使った時間差 (遅延) の評価

§ 6 の単発相互相関を **全イベント** に対して行い、ヒストグラムを作ります。
さらに精度を上げるため、**ピーク周辺 3 点で放物線フィット** をして **サブサンプル精度** で時間差を求めます。

### 放物線フィットによるサブサンプル補間

最大値の前後 3 点 `(−1, 0, +1)` の相関値を `y(−1), y(0), y(+1)` とすると、
真のピーク位置 (整数ピークからのオフセット) は次の式で近似できます:

$$ \delta = \frac{y(-1) - y(+1)}{2 \cdot (y(-1) - 2 y(0) + y(+1))} $$

これは「3 点を通る放物線」の頂点を求める初等的な式で、ノイズが小さければ ±0.1 サンプル程度の精度が出ます。


In [ ]:
def parabolic_peak_offset(y_m1: float, y_0: float, y_p1: float) -> float:
    '''3 点 (-1, 0, +1) の相関値から放物線フィットでピークのオフセットを返す。
    分母が 0 のときはオフセット 0 を返す (ピークが平坦)。'''
    denom = 2.0 * (y_m1 - 2.0 * y_0 + y_p1)
    if denom == 0:
        return 0.0
    return (y_m1 - y_p1) / denom

delays_ns = np.empty(n_shots)
for i in range(n_shots):
    a = ch1[i] - ch1[i].mean()
    b = ch2[i] - ch2[i].mean()
    # § 6 と同じ向きのラグ (正 = ch2 が遅延) になるように引数を (b, a) の順で渡す
    corr = np.correlate(b, a, mode="full")
    k = int(np.argmax(corr))
    if 0 < k < len(corr) - 1:
        offset = parabolic_peak_offset(corr[k - 1], corr[k], corr[k + 1])
    else:
        offset = 0.0
    lag_samples = (k - (len(a) - 1)) + offset       # 整数ラグ + サブサンプル補正
    delays_ns[i] = lag_samples * dt_s * 1e9

# ヒストグラム
fig = go.Figure()
fig.add_trace(go.Histogram(x=delays_ns, nbinsx=max(15, int(np.sqrt(n_shots))),
                           name="cross-corr delay"))
fig.update_layout(xaxis_title="時間差 ch2 - ch1 [ns]", yaxis_title="件数",
                  width=900, height=380,
                  title_text="相互相関による時間差 (サブサンプル精度)")
fig.show()

print(f"中央値:   {np.median(delays_ns):.3f} ns")
print(f"平均値:   {np.mean(delays_ns):.3f} ns")
print(f"標準偏差: {np.std(delays_ns):.3f} ns")

# § 4 の素朴 argmax 法と比較
delta_argmax_ns = (np.argmax(ch2, axis=1) - np.argmax(ch1, axis=1)) * dt_s * 1e9
print(f"\n[比較] 素朴 argmax 法の中央値: {np.median(delta_argmax_ns):.3f} ns "
      f"(±{np.std(delta_argmax_ns):.3f} ns)")

if known_delay is not None:
    expected_ns = float(known_delay) * dt_s * 1e9
    print(f"\n[テスト用] 既知の遅延: {expected_ns:.3f} ns "
          f"(差 = {np.median(delays_ns) - expected_ns:+.3f} ns)")


**読み取りのヒント**

- 素朴 `argmax` 法ではヒストグラムが **8 ns 刻みの離散値** になりますが、相互相関+放物線フィットでは **連続的な分布** になります。
- 中央値が 0 ns にならない場合は **ケーブル長の差** や **検出器の応答時間の違い** を表していることがあります。
- 分布の幅 (標準偏差) は **時間分解能** の指標。ここからジッタや信号波形の対称性を議論できます。


## おさらいと次の一歩

ここまでで、

- `.npz` の中身を **キー名** と **配列の形** から把握し、
- 波形を **時間軸 [μs]** で表示し、
- **平均化** で √N のノイズ低減を体感し、
- **argmax** と **相互相関** の 2 段階で時間差を評価しました。

ここから先のチャレンジ:

- ピーク値や面積で **しきい値カット** を掛けてから、信号らしいイベントだけで時間差を測り直してみよう。
- 全波形をフーリエ変換して **周波数スペクトル** を眺めてみよう (`np.fft.rfft`)。
- 波形の **立ち上がりエッジ** (例: 50% 高さ) で時間を取ると、`argmax` よりも安定した時間差が得られることがある。試してみよう (CFD = Constant Fraction Discriminator)。

## トラブルシューティング

### `samples_per_shot` が 16384 ではない
取得時に `DMA_DATA_SIZE` を変更した可能性があります。メタの `samples_per_shot` を信頼して使えば動作します。

### `.json` が見つからない
取得側ノートブックの古い版で取った `.npz` だと、メタ情報の `.json` が無いことがあります。その場合 `dt_s = 8e-9` (dec=1 を仮定) で続行できます。

### `np.load` が「遅い」
`.npz` は zip 圧縮なので、ファイルが大きいと展開に少し時間がかかります。`data["ch1"]` を呼んだ時点でその配列だけ展開されるので、不要なキーまで読まなくて済みます。

### 相互相関の中央値が物理的にあり得ない大きさになる
波形の長さに対してピークの位置がほぼ同じだと、ベースラインのノイズに相関のピークが乗ってしまい、本当のピークとは違う場所が選ばれることがあります。
ベースラインを引く (`a - a.mean()`) とともに、**ピーク周辺だけを切り出して相互相関を取る** と安定します (発展課題)。

### グラフが表示されない
JupyterLab の場合は次を実行: `import plotly.io as pio; pio.renderers.default = "jupyterlab"` (クラシック Notebook なら `"notebook_connected"`)。